# Landing → Bronze

Ingestão das 8 tabelas em um notebook só — a lógica é a mesma pra todas (ler fiel à origem, escrever
com `overwrite`), então não faz sentido uma cópia por pessoa como em `analises/`. **Mudar só por PR
revisado** (mesma regra do notebook base em `analises/analise_geral_corrigida.py`).

Regras da camada bronze (ver `CLAUDE.md`):
- **Fiel à origem**: tudo lido como `string`, nenhuma correção de conteúdo aqui (isso é papel da silver).
- **Idempotente**: escreve com `overwrite` — rodar de novo dá o mesmo resultado.
- **Rastreável**: toda tabela ganha `_ingestion_ts` e `_source_file`.
- **Sem dado pessoal em output**: `clientes` tem CPF/nome/e-mail — a verificação no final só mostra
  contagens, nunca `display()`/`show()` de linhas.

## 1. Configuração

In [ ]:
dbutils.widgets.text("base_path", "")
BASE_PATH = dbutils.widgets.get("base_path")
if not BASE_PATH:
    raise ValueError(
        "Preencha o widget 'base_path' com um caminho de Volume: '/Volumes/<catalogo>/<schema>/<volume>'."
    )
if not BASE_PATH.startswith("/Volumes"):
    raise ValueError(
        f"'{BASE_PATH}' não é um Volume. O Spark serverless não lê CSV de pasta do Workspace — "
        "suba os CSVs num Volume do Unity Catalog e use '/Volumes/<catalogo>/<schema>/<volume>'."
    )

CATALOGO = "workspace"
SCHEMA_BRONZE = "vcredit_bronze"

## 2. Fontes
Arquivo → (tabela de destino, encoding). `clientes.csv` e `Chamados_Hora.CSV` são `ISO-8859-1`; os
demais `UTF-8`. `Chamados_Hora.CSV` é a única fonte sem aspas.

In [ ]:
SOURCES = {
    "chamados.csv":            ("chamados",            "UTF-8"),
    "Chamados_Hora.CSV":       ("chamados_hora",       "ISO-8859-1"),
    "clientes.csv":            ("clientes",            "ISO-8859-1"),
    "custos.csv":              ("custos",              "UTF-8"),
    "pesquisa_satisfacao.csv": ("pesquisa_satisfacao", "UTF-8"),
    "base_atendentes.csv":     ("base_atendentes",     "UTF-8"),
    "base_motivos.csv":        ("base_motivos",        "UTF-8"),
    "canais.csv":              ("canais",              "UTF-8"),
}

## 3. Leitura (fiel à origem — tudo `string`) e escrita idempotente na bronze
`inferSchema=False` sempre: deixar o Spark adivinhar o tipo é arriscado (ex.: `id_cliente` é CPF com
zero à esquerda — se virar número, perde o zero e corrompe o dado).

In [ ]:
from pyspark.sql import functions as F

resumo = []

for arquivo, (tabela, encoding) in SOURCES.items():
    leitor = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("encoding", encoding)
    )
    if arquivo != "Chamados_Hora.CSV":
        leitor = leitor.option("quote", '"').option("escape", '"')

    df = leitor.csv(f"{BASE_PATH}/{arquivo}")

    # padroniza nomes de coluna (remove BOM/espaços à toa, minúsculo) — não muda o conteúdo das células
    for antigo in df.columns:
        novo = antigo.replace("\ufeff", "").strip().lower()
        if novo != antigo:
            df = df.withColumnRenamed(antigo, novo)

    df_bronze = (
        df
        .withColumn("_ingestion_ts", F.current_timestamp())
        .withColumn("_source_file", F.lit(arquivo))
    )

    destino = f"{CATALOGO}.{SCHEMA_BRONZE}.{tabela}"
    (
        df_bronze.write
        .mode("overwrite")               # idempotente: roda de novo, mesmo resultado
        .option("overwriteSchema", "true")
        .saveAsTable(destino)
    )

    linhas = spark.table(destino).count()
    resumo.append((tabela, arquivo, encoding, linhas))
    print(f"{destino}: {linhas:,} linhas")

## 4. Verificação
Contagens esperadas (arquivos originais, ver `docs/achados_analise_geral.md`): chamados 134.474 ·
chamados_hora 131.838 · clientes 47.705 · custos 131.069 · pesquisa_satisfacao 81.021 ·
base_atendentes 25 · base_motivos 14 · canais 4. Essa célula só mostra contagens — nunca linhas inteiras
(⚠️ `clientes` tem CPF/nome/e-mail).

In [ ]:
display(spark.createDataFrame(resumo, ["tabela", "arquivo", "encoding", "linhas"]))

## Checklist antes de comitar / dar push

- [ ] Contagens da seção 4 batem com o esperado.
- [ ] Nenhuma célula de output mostra linha de `clientes` (CPF/nome/e-mail).
- [ ] Rodei duas vezes e o resultado foi o mesmo (idempotente).
- [ ] Se precisou mudar algo, foi por PR revisado — não direto na `main`.